In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Workshop 3/excercises"

/content/drive/MyDrive/Workshop 3/excercises


In [3]:
import sys
sys.path.append("/content/drive/MyDrive/Workshop 3")

In [4]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
  3. GQA vs MHA — efecto de n_kv_heads
"""

'\nTaller: Internals de un LLM estilo LLaMA\n=================================================================\nInstrucciones:\n  - Busca los bloques marcados con TODO y completa el código.\n  - Cada sección tiene una celda de verificación al final.\n  - No modifiques nada fuera de los bloques TODO.\n\nSecciones:\n  3. GQA vs MHA — efecto de n_kv_heads\n'

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [6]:
# ===========================================================================
# SECCIÓN 3 — GQA vs MHA
# ===========================================================================
# Multi-Head Attention (MHA): n_kv_heads == n_heads  (sin compartir)
# Grouped Query Attention (GQA): n_kv_heads < n_heads (KV compartidos)
#
# Experimenta cambiando n_kv_heads y observa:
#   - Diferencia en parámetros de K y V
#   - Diferencia en los mapas de atención
# ===========================================================================

def build_attention(n_heads: int, n_kv_heads: int, d_model: int = 32) -> GroupedQueryAttention:
    # TODO 3.1 — Construye un GroupedQueryAttention con los parámetros dados
    cfg = ModelConfig(vocab_size=256, d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads)
    return GroupedQueryAttention(cfg)
    #raise NotImplementedError("TODO 3.1")


def compare_attention_maps(attn_mha, attn_gqa, x, rope_freqs, mask):
    # TODO 3.2 — Extrae los pesos de atención (scores softmax) de ambos módulos
    def forward(self, x: torch.Tensor,
                rope_freqs: torch.Tensor,
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, D = x.shape
        Dh = self.head_dim

        # Project to Q, K, V
        q = self.Wq(x).reshape(B, T, self.n_heads,    Dh)   # (B,T,Hq,Dh)
        k = self.Wk(x).reshape(B, T, self.n_kv_heads, Dh)   # (B,T,Hkv,Dh)
        v = self.Wv(x).reshape(B, T, self.n_kv_heads, Dh)   # (B,T,Hkv,Dh)

        # Apply RoPE to Q and K
        q = apply_rope(q, rope_freqs)
        k = apply_rope(k, rope_freqs)

        # Expand K, V to match Q heads: repeat each KV head n_rep times
        # (B, T, Hkv, Dh) → (B, T, Hq, Dh)
        k = k.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, Dh) \
             .reshape(B, T, self.n_heads, Dh)
        v = v.unsqueeze(3).expand(B, T, self.n_kv_heads, self.n_rep, Dh) \
             .reshape(B, T, self.n_heads, Dh)

        # Transpose for attention: (B, H, T, Dh)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled dot-product attention
        scale  = math.sqrt(Dh)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale   # (B,H,T,T)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn   = F.softmax(scores, dim=-1)

    attn_weights_mha = get_attn_weights(attn_mha, x, rope_freqs, mask)
    attn_weights_gqa = get_attn_weights(attn_gqa, x, rope_freqs, mask)

    return attn_weights_mha, attn_weights_gqa
    #raise NotImplementedError("TODO 3.2")


# ── Verificación 3 ──────────────────────────────────────────────────────────
def verify_section3():
    print("=" * 55)
    print("VERIFICACIÓN 3 — GQA vs MHA")
    print("=" * 55)
    D, Hq, Hkv, T = 32, 4, 2, 10

    mha = build_attention(n_heads=Hq,  n_kv_heads=Hq,  d_model=D)
    gqa = build_attention(n_heads=Hq,  n_kv_heads=Hkv, d_model=D)

    # Contar parámetros K+V
    mha_kv = mha.Wk.weight.numel() + mha.Wv.weight.numel()
    gqa_kv = gqa.Wk.weight.numel() + gqa.Wv.weight.numel()
    print(f"  MHA  K+V params : {mha_kv:,}")
    print(f"  GQA  K+V params : {gqa_kv:,}")
    print(f"  Reducción       : {(1 - gqa_kv/mha_kv)*100:.0f}%  (esperado {(1-Hkv/Hq)*100:.0f}%)")

    # Shapes del forward
    cfg        = ModelConfig(vocab_size=256, d_model=D, n_heads=Hq,
                             n_kv_heads=Hkv, d_ff=64, max_seq_len=T)
    x          = torch.randn(1, T, D)
    rope_freqs = precompute_rope_freqs(D // Hq, T)
    mask       = torch.tril(torch.ones(T, T))

    out_mha = build_attention(Hq, Hq, D)(x, rope_freqs, mask)
    out_gqa = build_attention(Hq, Hkv, D)(x, rope_freqs, mask)
    print(f"  MHA output shape: {list(out_mha.shape)}  (esperado [1, {T}, {D}])")
    print(f"  GQA output shape: {list(out_gqa.shape)}  (esperado [1, {T}, {D}])")

    shapes_ok = (out_mha.shape == out_gqa.shape == torch.Size([1, T, D]))
    kv_ok     = (gqa_kv == mha_kv * Hkv // Hq)
    if shapes_ok and kv_ok:
        print("\n  ✓ Sección 3 correcta")
    else:
        print("\n  ✗ Revisa tu implementación")

In [7]:
verify_section3()

VERIFICACIÓN 3 — GQA vs MHA
  MHA  K+V params : 2,048
  GQA  K+V params : 1,024
  Reducción       : 50%  (esperado 50%)
  MHA output shape: [1, 10, 32]  (esperado [1, 10, 32])
  GQA output shape: [1, 10, 32]  (esperado [1, 10, 32])

  ✓ Sección 3 correcta
